# Авито: кандидаты BM25 + dense (энкодер) и признаки — v2

**Входы (Add Input):** результат предобработки (`prep/`) и вывод ноутбука энкодера (`encoder/`: `item_emb.npy`, `item_title_emb.npy`,
`query_emb.npy`, `item_ids.parquet`, `query_ids.parquet`). CPU-сессия, GPU не нужен.

**Что нового по сравнению с v1:**
1. **Третий источник кандидатов**: топ-`K_DENSE` по косинусу энкодера внутри того же гео-пула. На val это поднимает потолок с 0.932 до ~0.967
2. **Признаки энкодера** для всех кандидатов:
   - `cos_full`: косинус запроса и полного текста объявления (заголовок + параметры + описание)
   - `cos_title`: косинус запроса и заголовка
   - `rank_dense`: место кандидата в топе по `cos_full` внутри пула (`K_DENSE` = не попал в топ)
   - `cos_gap`: насколько `cos_full` кандидата ниже лучшего косинуса в пуле этого запроса
3. **Бенчмарк**: те же кандидаты и признаки для 2452 запросов бенчмарка (`bench.parquet`, без `target`), пригодится для `answer.csv`
4. Результат в папке `features_v2/` с файлом `features_v2.json`, чтобы не путать со старыми признаками во входах

Запросы для CatBoost-train выбираются с тем же `SEED`, что и в v1: это те же 15 тыс. запросов, на которых классификатор
делал out-of-fold предсказания и которые энкодер не видел при обучении.

### Импорты и параметры

In [1]:
import os, re, gc, glob, json, time, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', 60)

def find_dir(filename, roots=('/kaggle/input', '/kaggle/working', '.')):
    """Папка, где лежит filename: сначала подключённые входы (/kaggle/input), потом рабочая папка."""
    for root in roots:
        hits = sorted(glob.glob(os.path.join(root, '**', filename), recursive=True))
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError(f'{filename} не найден ни в /kaggle/input, ни в /kaggle/working')

WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

PREP_DIR = find_dir('pairs.parquet')          # результат предобработки: вход ноутбука или /kaggle/working/prep
ENC_DIR = find_dir('item_emb.npy')             # вывод ноутбука энкодера
FEAT_DIR = f'{WORK}/features_v2'               # сюда пишем результат
os.makedirs(FEAT_DIR, exist_ok=True)

SEED = 42
GEO_RADIUS_KM = 50       # гео-пул: тот же id ИЛИ ближе этого радиуса
K_POOL = 200             # кандидатов из гео-пула
K_GLOBAL = 50            # кандидатов по всему корпусу без гео (дальние выборы)
K_DENSE = 100            # кандидатов по косинусу энкодера внутри гео-пула
MAX_TRAIN_Q = 15_000     # сколько train-запросов берём для CatBoost (хватит с запасом)
BATCH = 128              # запросов за один проход BM25
BM25_K1, BM25_B = 1.2, 0.75
TITLE_WEIGHT = 2         # в общем поле заголовок учитывается дважды: он точнее описания

try:
    import psutil
    def mem(tag=''):
        print(f'[mem] {tag}: {psutil.Process().memory_info().rss / 2**30:.1f} ГБ')
except ImportError:
    def mem(tag=''): pass
print('prep:', PREP_DIR, '| encoder:', ENC_DIR, '| вывод:', FEAT_DIR)

prep: /kaggle/input/notebooks/olegkaran/avito-kandidates-ranking/prep | encoder: /kaggle/input/notebooks/olegkaran/encoder-finetune-v2/encoder | вывод: /kaggle/working/features_v2


## 1. Загрузка

Из `items.parquet` (v1) читаем **только объявления корпуса** и только нужные колонки. Сырые тексты и `desc_norm` не грузим,
иначе снова упрёмся в память.

In [2]:
ITEM_COLS = ['item_id', 'item_category_id', 'item_microcat_id', 'item_location_id', 'lat', 'lon', 'is_remote_text',
             'item_rating', 'rating_isna', 'reviews_log', 'price_log', 'price_is_symbolic',
             'item_is_phone_hidden', 'item_is_message_forbidden',
             'title_norm', 'params_norm', 'title_lem', 'params_lem', 'desc_lem']
items = pd.read_parquet(f'{PREP_DIR}/items.parquet', columns=ITEM_COLS,
                        filters=[('in_corpus', '==', True)]).reset_index(drop=True)
for c in ['title_lem', 'params_lem', 'desc_lem', 'params_norm', 'title_norm']:
    items[c] = items[c].fillna('')

queries = pd.read_parquet(f'{PREP_DIR}/queries.parquet',
                          columns=['qid', 'split', 'search_query', 'query_norm', 'query_lem', 'search_location_id',
                                   'search_category', 'search_infm_params_text', 'loc_lat', 'loc_lon', 'loc_n_items'])
pairs = pd.read_parquet(f'{PREP_DIR}/pairs.parquet')
print(f'корпус: {len(items):,} | запросов: {len(queries):,} | пар: {len(pairs):,}')
mem('после загрузки')

корпус: 189,212 | запросов: 356,914 | пар: 467,054
[mem] после загрузки: 2.4 ГБ


## 2. Выбор запросов и их релевантных объявлений

Релевантные для запроса — его выбранные объявления, **которые есть в корпусе** (остальные найти невозможно).
`n_pos_total` — знаменатель Recall для запроса; он сохраняется в файл, чтобы метрика считалась честно,
даже если релевантное объявление не попало в кандидаты.

In [3]:
id2idx = pd.Series(np.arange(len(items)), index=items['item_id'])
pos_pairs = pairs[pairs.in_corpus & pairs.item_id.isin(id2idx.index)]
POS = pos_pairs.groupby('qid')['item_id'].apply(lambda s: set(id2idx[s].values)).to_dict()   # qid -> {индексы строк}

val_q = queries[(queries.split == 'val') & queries.qid.isin(POS)].reset_index(drop=True)
tr_all = queries[(queries.split == 'train') & queries.qid.isin(POS)]
tr_q = tr_all.sample(n=min(MAX_TRAIN_Q, len(tr_all)), random_state=SEED).reset_index(drop=True)
print(f'train-запросов с объявлением в корпусе: {len(tr_all):,}, берём {len(tr_q):,}')
bench_q = queries[queries.split == 'bench'].reset_index(drop=True)
print(f'val-запросов: {len(val_q):,} | бенчмарк: {len(bench_q):,}')
print(f'бенчмарк без оценки координат локации: {bench_q.loc_lat.isna().mean():.2%} запросов (у них гео-фильтр не применяется)')
print(f'релевантных на запрос: train {np.mean([len(POS[q]) for q in tr_q.qid]):.2f}, val {np.mean([len(POS[q]) for q in val_q.qid]):.2f}')

train-запросов с объявлением в корпусе: 23,363, берём 15,000
val-запросов: 3,000 | бенчмарк: 2,452
бенчмарк без оценки координат локации: 0.00% запросов (у них гео-фильтр не применяется)
релевантных на запрос: train 1.12, val 1.02


## 3. BM25 по полям

Считаем словарь по леммам всех трёх полей, затем для каждого поля строим матрицу весов BM25 размером «объявления × слова»:

$w(t, d) = idf(t) \cdot \frac{tf(t,d)(k_1+1)}{tf(t,d) + k_1(1 - b + b\cdot|d|/avgdl)}$

Скор запроса — сумма весов его слов, то есть одно умножение разреженной матрицы на вектор запроса.
Поле `all` = заголовок ×2 + параметры + описание; по нему отбираются кандидаты. Скоры по отдельным полям идут признаками.

In [4]:
def bm25_matrix(tf, k1=BM25_K1, b=BM25_B):
    """Матрица tf (объявления × слова) -> матрица весов BM25 той же формы (CSR)."""
    tf = tf.tocsr().astype(np.float32)
    n_docs = tf.shape[0]
    dl = np.asarray(tf.sum(axis=1)).ravel()
    avgdl = max(dl.mean(), 1e-9)
    df = np.bincount(tf.indices, minlength=tf.shape[1])
    idf = np.log(1 + (n_docs - df + 0.5) / (df + 0.5)).astype(np.float32)
    rows = np.repeat(np.arange(n_docs), np.diff(tf.indptr))
    data = tf.data * (k1 + 1) / (tf.data + k1 * (1 - b + b * dl[rows] / avgdl)) * idf[tf.indices]
    return sp.csr_matrix((data.astype(np.float32), tf.indices, tf.indptr), shape=tf.shape)

t = time.time()
cv = CountVectorizer(token_pattern=r'\S+', lowercase=False, dtype=np.float32)
n = len(items)
# один проход токенизации по трём полям сразу, потом режем по строкам
X = cv.fit_transform(pd.concat([items.title_lem, items.params_lem, items.desc_lem], ignore_index=True)).tocsr()
TF = {'title': X[:n], 'params': X[n:2 * n], 'desc': X[2 * n:]}
del X
TF['all'] = (TITLE_WEIGHT * TF['title'] + TF['params'] + TF['desc']).tocsr()

W = {f: bm25_matrix(TF[f]) for f in ['all', 'title', 'params', 'desc']}
# транспонированная матрица для быстрого скоринга батча: (запросы × слова) @ (слова × объявления)
# читает только строки слов, которые есть в запросах, а не всю матрицу
WT_ALL = W['all'].T.tocsr()
BIN = {f: (TF[f] > 0).astype(np.float32).tocsr() for f in ['title', 'all']}   # «есть ли слово в поле», для покрытия

item_len = {f'{f}_len': np.asarray(TF[f].sum(axis=1)).ravel().astype(np.float32) for f in ['title', 'params', 'desc']}
del TF; gc.collect()
print(f'словарь: {len(cv.vocabulary_):,} слов | {time.time() - t:.0f} с')
mem('после BM25-матриц')

словарь: 399,994 слов | 45 с
[mem] после BM25-матриц: 3.3 ГБ


## 3б. Векторы энкодера

Векторы уже нормированы, поэтому косинус = скалярное произведение. Переставляем строки векторов в порядок объявлений
этого ноутбука по `item_ids.parquet` (не полагаемся на то, что порядок совпадает сам). Считаем в float32 (~0.6 ГБ на матрицу).

In [5]:
enc_items = pd.read_parquet(f'{ENC_DIR}/item_ids.parquet')['item_id']
enc_pos = pd.Series(np.arange(len(enc_items)), index=enc_items.values)
assert items['item_id'].isin(enc_pos.index).all(), 'в векторах энкодера есть не все объявления корпуса'
order = enc_pos[items['item_id']].values
E_FULL = np.load(f'{ENC_DIR}/item_emb.npy')[order].astype(np.float32)
E_TITLE = np.load(f'{ENC_DIR}/item_title_emb.npy')[order].astype(np.float32)

enc_q = pd.read_parquet(f'{ENC_DIR}/query_ids.parquet')['qid']
Q_POS = pd.Series(np.arange(len(enc_q)), index=enc_q.values)
Q_EMB = np.load(f'{ENC_DIR}/query_emb.npy').astype(np.float32)
for name, qd in [('train', tr_q), ('val', val_q), ('bench', bench_q)]:
    miss = (~qd.qid.isin(Q_POS.index)).sum()
    assert miss == 0, f'{name}: у {miss} запросов нет вектора — энкодер считался на другом наборе запросов'
print(f'векторы: объявления {E_FULL.shape}, запросы {Q_EMB.shape}')
mem('после загрузки векторов')

векторы: объявления (189212, 768), запросы (20452, 768)
[mem] после загрузки векторов: 4.5 ГБ


## 4. Функции фильтров (исправленная логика из v2)

Фильтр поиска покрывается кусками, которые дословно встречаются в `params_norm` объявления. Рейтинг из фильтра проверяется
отдельно как `item_rating >= N`. Фильтр запроса нормализуется той же функцией, что и параметры объявления в v1.

In [6]:
URL_RE   = re.compile(r'https?://\S+|www\.\S+')
EMAIL_RE = re.compile(r'\S+@\S+\.\S+')
PHONE_RE = re.compile(r'(?:\+7|\b8)[\s\-(]*\d{3}[\s\-)]*\d{3}[\s\-]*\d{2}[\s\-]*\d{2}')
NOISE_RE = re.compile(r'[^\w\s.,:;!?()«»"\'%№/+\-]')
WS_RE    = re.compile(r'\s+')
RATING_RE = re.compile(r'Рейтинг пользователя (\d)\S*\s+звезд\S*\s+и\s+выше')

def normalize_for_model(s):
    if not isinstance(s, str) or not s:
        return ''
    s = s.replace('\\n', ' ').replace('ё', 'е').replace('Ё', 'Е')
    s = URL_RE.sub(' ', s); s = EMAIL_RE.sub(' ', s); s = PHONE_RE.sub(' ', s); s = NOISE_RE.sub(' ', s)
    return WS_RE.sub(' ', s).strip()

def parse_filter(f):
    """-> (минимальный рейтинг или NaN, текстовая часть фильтра)."""
    f = normalize_for_model(f)
    m = RATING_RE.search(f)
    return (float(m.group(1)) if m else np.nan), WS_RE.sub(' ', RATING_RE.sub(' ', f)).strip()

def covered(f, params):
    """Можно ли разрезать фильтр на куски подряд идущих слов, каждый из которых есть в params."""
    w = f.split(' ')
    i, n_ = 0, len(w)
    while i < n_:
        j = i
        while j < n_ and ' '.join(w[i:j + 1]) in params:
            j += 1
        if j == i:
            return False
        i = j
    return True

print(parse_filter('Рейтинг пользователя 4 звезды и выше Вид услуги Красота, здоровье'))

(4.0, 'Вид услуги Красота, здоровье')


## 5. Гео и категория: маски пула

Для каждого запроса пул = (категория подходит) И (тот же `location_id` ИЛИ расстояние ≤ `GEO_RADIUS_KM`).
Если координаты локации неизвестны, гео-фильтр не применяется. Если пул пустой, берётся весь корпус подходящей категории.

In [7]:
I_LAT = np.radians(items.lat.values.astype('float64'))
I_LON = np.radians(items.lon.values.astype('float64'))
I_LOC = items.item_location_id.values
I_REMOTE = items.is_remote_text.values.astype(bool)
CAT_MASK = {114: items.item_category_id.values == 114}
ALL_TRUE = np.ones(len(items), dtype=bool)
I_MC = items.item_microcat_id.values
I_PARAMS = items.params_norm.values
I_RATING = items.item_rating.values.astype('float64')

def dist_km(lat, lon):
    """Расстояние от точки (градусы) до всех объявлений корпуса, км."""
    lat, lon = np.radians(lat), np.radians(lon)
    a = np.sin((I_LAT - lat) / 2) ** 2 + np.cos(lat) * np.cos(I_LAT) * np.sin((I_LON - lon) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def top_k(idx, scores, k):
    """k индексов из idx с наибольшим скором, отсортированные по убыванию."""
    if len(idx) == 0 or k <= 0:
        return idx[:0]
    k = min(k, len(idx))
    part = idx[np.argpartition(-scores[idx], k - 1)[:k]]
    return part[np.argsort(-scores[part], kind='stable')]

## 6. Кандидаты и признаки

Для каждого запроса:
- **пул**: `K_POOL` лучших по BM25 в пуле плюс `K_GLOBAL` лучших по корпусу категории, объединённые
- **признаки** (все считаются одинаково для train и val):

| группа | признаки |
|---|---|
| BM25 | `bm25_all`, `bm25_title`, `bm25_params`, `bm25_desc`, `bm25_rel` (скор / лучший скор в пуле), `rank_pool`, `rank_global`, `in_pool` |
| покрытие запроса | `cov_title`, `cov_all`: доля слов запроса, найденных в заголовке / во всех полях |
| микрокатегория | `mc_share_top20`: доля микрокатегории кандидата среди топ-20 BM25 запроса (по сути, «о чём этот запрос») |
| гео | `dist_km`, `same_loc`, `is_remote_text` |
| фильтр | `has_filter`, `filter_text_ok`, `rating_ok`, `filter_ok` |
| объявление | рейтинг, отзывы, цена, флаги, длины полей, `item_microcat_id` (категориальный) |
| запрос | `q_n_terms`, `q_len_words`, `search_category`, `loc_n_items`, `pool_size`, `geo_known` |

Служебные колонки (не признаки): `qid`, `item_id`, `target`, `n_pos_total`.

In [8]:
def build_features(qdf, name):
    t0 = time.time()
    Q = cv.transform(qdf['query_lem'].fillna('')).tocsr()
    Q.data[:] = 1                                    # важно наличие слова в запросе, а не частота
    q_n_terms = np.asarray(Q.sum(axis=1)).ravel()
    filters = [parse_filter(f) for f in qdf['search_infm_params_text'].fillna('')]
    cols = {k: [] for k in ['qid_idx', 'cand', 'bm25_all', 'bm25_title', 'bm25_params', 'bm25_desc', 'bm25_rel',
                            'rank_pool', 'rank_global', 'in_pool', 'cov_title', 'cov_all', 'mc_share_top20',
                            'dist_km', 'same_loc', 'filter_text_ok', 'rating_ok', 'filter_ok', 'pool_size', 'target',
                            'cos_full', 'cos_title', 'rank_dense', 'cos_gap']}
    stats = {'cand_recall': [], 'cand_recall_bm25_only': [], 'bm25_pool_recall50': [], 'dense_pool_recall50': []}
    QE = Q_EMB[Q_POS[qdf['qid']].values]              # векторы запросов в порядке qdf

    for b0 in range(0, len(qdf), BATCH):
        qb = Q[b0:b0 + BATCH]
        S = (qb @ WT_ALL).toarray()                           # (запросы батча × корпус): BM25 по полю all
        DC = QE[b0:b0 + BATCH] @ E_FULL.T                     # (запросы батча × корпус): косинус энкодера
        for j in range(qb.shape[0]):
            qi = b0 + j
            row = qdf.iloc[qi]
            s = S[j]
            cat = CAT_MASK.get(int(row.search_category), ALL_TRUE)

            # --- гео-пул
            if pd.notna(row.loc_lat):
                d = dist_km(row.loc_lat, row.loc_lon)
                # удалёнку в пул НЕ добавляем: флаг is_remote_text срабатывает слишком широко и раздувает пул до половины корпуса;
                # он остаётся признаком, а дальние выборы ловит источник BM25 по всему корпусу
                geo = (I_LOC == row.search_location_id) | (d <= GEO_RADIUS_KM)
            else:
                d = np.full(len(items), np.nan)
                geo = ALL_TRUE
            pool_mask = cat & geo
            if not pool_mask.any():
                pool_mask = cat
            pool_idx = np.flatnonzero(pool_mask)

            # --- кандидаты из трёх источников: BM25 в пуле, BM25 по корпусу, энкодер в пуле
            dc = DC[j]
            top_pool = top_k(pool_idx, s, K_POOL)
            top_glob = top_k(np.flatnonzero(cat), s, K_GLOBAL)
            top_dense = top_k(pool_idx, dc, K_DENSE)
            cand = np.unique(np.concatenate([top_pool, top_glob, top_dense]))
            rank_dense = pd.Series(np.arange(len(top_dense)), index=top_dense).reindex(cand).fillna(K_DENSE).values
            rank_pool = pd.Series(np.arange(len(top_pool)), index=top_pool).reindex(cand).fillna(K_POOL).values
            rank_glob = pd.Series(np.arange(len(top_glob)), index=top_glob).reindex(cand).fillna(K_GLOBAL).values

            # --- скоры по полям и покрытие слов запроса (только для кандидатов)
            qv = qb[j].T
            f_title  = np.asarray((W['title'][cand] @ qv).todense()).ravel()
            f_params = np.asarray((W['params'][cand] @ qv).todense()).ravel()
            f_desc   = np.asarray((W['desc'][cand] @ qv).todense()).ravel()
            nt = max(q_n_terms[qi], 1)
            c_title = np.asarray((BIN['title'][cand] @ qv).todense()).ravel() / nt
            c_all   = np.asarray((BIN['all'][cand] @ qv).todense()).ravel() / nt

            # --- «о чём запрос»: распределение микрокатегорий в топ-20 пула
            mc_top = pd.Series(I_MC[top_pool[:20]]).value_counts(normalize=True)
            mc_share = pd.Series(I_MC[cand]).map(mc_top).fillna(0).values

            # --- фильтр
            min_r, rest = filters[qi]
            ftxt = np.array([covered(rest, I_PARAMS[c]) for c in cand]) if rest else np.ones(len(cand), bool)
            r_ok = (I_RATING[cand] >= min_r) if min_r == min_r else np.ones(len(cand), bool)

            best = s[top_pool[0]] if len(top_pool) else 0.0
            best_cos = dc[top_dense[0]] if len(top_dense) else 0.0
            pos = POS.get(row.qid, set())                    # у бенчмарка ответов нет -> пустое множество
            tgt = np.array([c in pos for c in cand])

            cols['qid_idx'].append(np.full(len(cand), qi)); cols['cand'].append(cand)
            cols['bm25_all'].append(s[cand]); cols['bm25_title'].append(f_title)
            cols['bm25_params'].append(f_params); cols['bm25_desc'].append(f_desc)
            cols['bm25_rel'].append(s[cand] / best if best > 0 else np.zeros(len(cand)))
            cols['rank_pool'].append(rank_pool); cols['rank_global'].append(rank_glob)
            cols['in_pool'].append(pool_mask[cand]); cols['cov_title'].append(c_title); cols['cov_all'].append(c_all)
            cols['mc_share_top20'].append(mc_share)
            cols['dist_km'].append(d[cand]); cols['same_loc'].append(I_LOC[cand] == row.search_location_id)
            cols['filter_text_ok'].append(ftxt); cols['rating_ok'].append(r_ok); cols['filter_ok'].append(ftxt & r_ok)
            cols['pool_size'].append(np.full(len(cand), len(pool_idx))); cols['target'].append(tgt)
            cols['cos_full'].append(dc[cand]); cols['cos_title'].append(E_TITLE[cand] @ QE[qi])
            cols['rank_dense'].append(rank_dense); cols['cos_gap'].append(best_cos - dc[cand])

            if pos:
                stats['cand_recall'].append(tgt.sum() / len(pos))
                stats['cand_recall_bm25_only'].append(len(pos & (set(top_pool) | set(top_glob))) / len(pos))
                stats['bm25_pool_recall50'].append(len(set(top_pool[:50]) & pos) / len(pos))
                stats['dense_pool_recall50'].append(len(set(top_dense[:50]) & pos) / len(pos))

        if (b0 // BATCH) % 20 == 0:
            print(f'  {name}: {min(b0 + BATCH, len(qdf)):,}/{len(qdf):,} запросов, {time.time() - t0:.0f} с')

    df = pd.DataFrame({k: np.concatenate(v) for k, v in cols.items()})
    qi, ci = df.pop('qid_idx').values, df.pop('cand').values

    # --- служебные колонки
    df.insert(0, 'qid', qdf['qid'].values[qi])
    df.insert(1, 'item_id', items['item_id'].values[ci])
    df['n_pos_total'] = qdf['qid'].map(lambda q: len(POS.get(q, ()))).values[qi]

    # --- признаки запроса
    df['q_n_terms'] = q_n_terms[qi]
    df['q_len_words'] = qdf['query_norm'].fillna('').str.split().str.len().values[qi]
    df['search_category'] = qdf['search_category'].values[qi]
    df['loc_n_items'] = qdf['loc_n_items'].values[qi]
    df['geo_known'] = qdf['loc_lat'].notna().values[qi]
    df['has_filter'] = np.array([bool(r) or m == m for m, r in filters])[qi]

    # --- признаки объявления
    for c in ['item_microcat_id', 'is_remote_text', 'item_rating', 'rating_isna', 'reviews_log', 'price_log',
              'price_is_symbolic', 'item_is_phone_hidden', 'item_is_message_forbidden']:
        df[c] = items[c].values[ci]
    for c, v in item_len.items():
        df[c] = v[ci]

    # --- компактные типы
    for c in df.columns:
        if df[c].dtype == bool:
            df[c] = df[c].astype('int8')
        elif df[c].dtype == np.float64:
            df[c] = df[c].astype('float32')
    df['target'] = df['target'].astype('int8')

    print(f'{name}: {len(qdf):,} запросов, {len(df):,} строк, {time.time() - t0:.0f} с')
    return df, {k: (float(np.mean(v)) if v else float('nan')) for k, v in stats.items()}

val_df, val_stats = build_features(val_q, 'val')
mem('после val')

  val: 128/3,000 запросов, 3 с
  val: 2,688/3,000 запросов, 68 с
val: 3,000 запросов, 857,038 строк, 77 с
[mem] после val: 4.9 ГБ


In [9]:
train_df, train_stats = build_features(tr_q, 'train')
mem('после train')
bench_df, _ = build_features(bench_q, 'bench')
bench_df['target'] = np.int8(-1)
mem('после бенчмарка')

  train: 128/15,000 запросов, 4 с
  train: 2,688/15,000 запросов, 68 с
  train: 5,248/15,000 запросов, 132 с
  train: 7,808/15,000 запросов, 197 с
  train: 10,368/15,000 запросов, 261 с
  train: 12,928/15,000 запросов, 327 с
train: 15,000 запросов, 4,269,507 строк, 381 с
[mem] после train: 6.0 ГБ
  bench: 128/2,452 запросов, 3 с
bench: 2,452 запросов, 689,558 строк, 64 с
[mem] после бенчмарка: 6.1 ГБ


## 7. Сохранение и проверка, что наборы признаков совпадают

In [10]:
SERVICE_COLS = ['qid', 'item_id', 'target', 'n_pos_total']
CAT_FEATURES = ['item_microcat_id', 'search_category']
FEATURES = [c for c in train_df.columns if c not in SERVICE_COLS]

COS_FEATURES = ['cos_full', 'cos_title', 'rank_dense', 'cos_gap']
assert list(train_df.columns) == list(val_df.columns) == list(bench_df.columns), 'наборы колонок различаются!'
train_df.to_parquet(f'{FEAT_DIR}/train.parquet', index=False)
val_df.to_parquet(f'{FEAT_DIR}/val.parquet', index=False)
bench_df.to_parquet(f'{FEAT_DIR}/bench.parquet', index=False)
with open(f'{FEAT_DIR}/features_v2.json', 'w') as f:
    json.dump({'features': FEATURES, 'cat_features': CAT_FEATURES, 'cos_features': COS_FEATURES, 'service': SERVICE_COLS,
               'params': {'GEO_RADIUS_KM': GEO_RADIUS_KM, 'K_POOL': K_POOL, 'K_GLOBAL': K_GLOBAL, 'K_DENSE': K_DENSE}},
              f, ensure_ascii=False, indent=2)
print(f'признаков: {len(FEATURES)} (категориальные: {CAT_FEATURES})')
print(FEATURES)
for f_ in sorted(os.listdir(FEAT_DIR)):
    print(f'{f_:<16} {os.path.getsize(os.path.join(FEAT_DIR, f_)) / 2**20:7.1f} МБ')

признаков: 39 (категориальные: ['item_microcat_id', 'search_category'])
['bm25_all', 'bm25_title', 'bm25_params', 'bm25_desc', 'bm25_rel', 'rank_pool', 'rank_global', 'in_pool', 'cov_title', 'cov_all', 'mc_share_top20', 'dist_km', 'same_loc', 'filter_text_ok', 'rating_ok', 'filter_ok', 'pool_size', 'cos_full', 'cos_title', 'rank_dense', 'cos_gap', 'q_n_terms', 'q_len_words', 'search_category', 'loc_n_items', 'geo_known', 'has_filter', 'item_microcat_id', 'is_remote_text', 'item_rating', 'rating_isna', 'reviews_log', 'price_log', 'price_is_symbolic', 'item_is_phone_hidden', 'item_is_message_forbidden', 'title_len', 'params_len', 'desc_len']
bench.parquet       41.2 МБ
features_v2.json     0.0 МБ
train.parquet      241.1 МБ
val.parquet         50.2 МБ


## 8. Первые метрики

- **Потолок Recall**: доля релевантных, которые вообще попали в кандидаты. Выше него CatBoost не прыгнет
- **BM25 в пуле, топ-50**: бейзлайн без всякого обучения, просто 50 лучших по BM25 в гео-пуле. Это та планка, которую должен побить CatBoost

In [11]:
summary = pd.DataFrame({
    'train': [train_df.qid.nunique(), len(train_df), len(train_df) / train_df.qid.nunique(), train_df.target.mean(),
              train_stats['cand_recall'], train_stats['bm25_pool_recall50']],
    'val':   [val_df.qid.nunique(), len(val_df), len(val_df) / val_df.qid.nunique(), val_df.target.mean(),
              val_stats['cand_recall'], val_stats['bm25_pool_recall50']],
}, index=['запросов', 'строк', 'кандидатов на запрос', 'доля target=1',
          'потолок Recall (релевантное в кандидатах)', 'Recall@50: BM25 в пуле (бейзлайн)'])
summary.loc['потолок только по кандидатам BM25 (как в v1)'] = [train_stats['cand_recall_bm25_only'], val_stats['cand_recall_bm25_only']]
summary.loc['Recall@50: энкодер в пуле'] = [train_stats['dense_pool_recall50'], val_stats['dense_pool_recall50']]
display(summary.style.format('{:,.4f}'))
print(f'бенчмарк: {bench_df.qid.nunique():,} запросов, {len(bench_df):,} строк, {len(bench_df) / bench_df.qid.nunique():.1f} кандидатов на запрос')

# Проверка на утечку в косинус: энкодер не видел запросы CatBoost-train, но мог видеть такие же тексты из других городов.
# Если энкодер на train заметно «лучше», чем BM25 на train (относительно val), CatBoost будет переоценивать косинус.
r_bm, r_dn = train_stats['bm25_pool_recall50'] / val_stats['bm25_pool_recall50'], train_stats['dense_pool_recall50'] / val_stats['dense_pool_recall50']
print(f'отношение train/val: BM25 {r_bm:.3f}, энкодер {r_dn:.3f} '
      f'({"похоже, утечки нет" if r_dn - r_bm < 0.02 else "энкодер на train заметно лучше — возможна утечка через одинаковые тексты"})')

,train,val
запросов,"15,000.0000","3,000.0000"
строк,"4,269,507.0000","857,038.0000"
кандидатов на запрос,284.6338,285.6793
доля target=1,0.0038,0.0034
потолок Recall (релевантное в кандидатах),0.9719,0.9693
Recall@50: BM25 в пуле (бейзлайн),0.8709,0.8339
потолок только по кандидатам BM25 (как в v1),0.9495,0.9322
Recall@50: энкодер в пуле,0.8858,0.8406


бенчмарк: 2,452 запросов, 689,558 строк, 281.2 кандидатов на запрос
отношение train/val: BM25 1.044, энкодер 1.054 (похоже, утечки нет)


## 9. Как выглядят данные

In [12]:
print('train: первые 10 строк')
display(train_df.head(10))
print('val: первые 10 строк')
display(val_df.head(10))

train: первые 10 строк


,qid,item_id,bm25_all,bm25_title,bm25_params,bm25_desc,bm25_rel,rank_pool,rank_global,in_pool,cov_title,cov_all,mc_share_top20,dist_km,same_loc,filter_text_ok,rating_ok,filter_ok,pool_size,target,cos_full,cos_title,rank_dense,cos_gap,n_pos_total,q_n_terms,q_len_words,search_category,loc_n_items,geo_known,has_filter,item_microcat_id,is_remote_text,item_rating,rating_isna,reviews_log,price_log,price_is_symbolic,item_is_phone_hidden,item_is_message_forbidden,title_len,params_len,desc_len
0,tr_caceaad574bb3bc4,cbc5d40a256e7685,2.579009,2.090788,2.303012,3.275523,0.172206,69.0,50.0,1,0.5,0.5,0.00,7.335962,1,1,1,1,3851,0,0.070871,0.071827,100.0,0.601205,1,2.0,2,114,9807,1,0,84438,0,5.000000,0,5.187386,6.216606,0,0,0,5.0,162.0,138.0
1,tr_caceaad574bb3bc4,e70427e475b714fb,0.000000,0.000000,0.000000,0.000000,0.000000,200.0,50.0,1,0.0,0.0,0.00,1.148797,1,1,1,1,3851,0,0.325237,0.275109,54.0,0.346839,1,2.0,2,114,9807,1,0,1289831,0,4.500000,0,1.945910,8.006701,0,0,0,3.0,89.0,46.0
2,tr_caceaad574bb3bc4,4cd525d3f3a7d82b,13.415874,0.000000,0.000000,12.205449,0.895807,200.0,23.0,0,0.0,0.5,0.00,726.337524,0,1,1,1,3851,0,0.281693,0.277128,100.0,0.390383,1,2.0,2,114,9807,1,0,1289819,0,NaN,1,1.098612,8.294300,0,0,0,4.0,22.0,223.0
3,tr_caceaad574bb3bc4,76c424310652d5b4,2.335919,2.304721,1.879228,2.239165,0.155974,200.0,50.0,1,0.5,0.5,0.15,7.065244,1,1,1,1,3851,0,0.490004,0.193942,9.0,0.182072,1,2.0,2,114,9807,1,0,2169983,0,5.000000,0,1.945910,6.908755,0,0,0,4.0,73.0,44.0
4,tr_caceaad574bb3bc4,32da95bb8db64a54,0.000000,0.000000,0.000000,0.000000,0.000000,200.0,50.0,1,0.0,0.0,0.00,3.380656,1,1,1,1,3851,0,0.372952,0.348354,39.0,0.299124,1,2.0,2,114,9807,1,0,1289833,0,4.942623,0,6.192362,6.908755,0,0,0,5.0,138.0,30.0
5,tr_caceaad574bb3bc4,c807cf48af140e98,2.536615,2.304721,1.864747,3.136667,0.169375,104.0,50.0,1,0.5,0.5,0.05,37.871632,0,1,1,1,3851,0,0.034367,0.049613,100.0,0.637709,1,2.0,2,114,9807,1,0,2169975,0,4.200000,0,1.791759,6.398595,0,0,0,4.0,75.0,132.0
6,tr_caceaad574bb3bc4,48d6c714ed99e368,14.879358,10.040675,1.836444,10.268937,0.993527,200.0,10.0,0,1.0,1.0,0.15,1195.091064,0,1,1,1,3851,0,0.697757,0.554059,100.0,-0.025681,1,2.0,2,114,9807,1,0,2169983,1,4.921568,0,3.951244,5.707110,0,0,0,5.0,79.0,182.0
7,tr_caceaad574bb3bc4,44d055aa287eb7f7,2.654445,2.090788,1.947276,3.344653,0.177243,18.0,50.0,1,0.5,0.5,0.10,1.324593,1,1,1,1,3851,0,0.105363,0.081001,100.0,0.566713,1,2.0,2,114,9807,1,0,2169973,0,4.866667,0,3.828641,6.175867,0,0,0,5.0,64.0,350.0
8,tr_caceaad574bb3bc4,653002c3a8449531,2.452737,2.304721,2.537263,2.678664,0.163775,187.0,50.0,1,0.5,0.5,0.00,6.681722,1,1,1,1,3851,0,0.274763,0.324921,88.0,0.397313,1,2.0,2,114,9807,1,0,84438,0,4.902778,0,4.290460,0.000000,1,0,0,4.0,55.0,104.0
9,tr_caceaad574bb3bc4,fabab8148cd6e4fe,1.618586,0.000000,1.916434,0.000000,0.108077,200.0,50.0,1,0.0,0.5,0.15,4.748390,1,1,1,1,3851,0,0.397356,0.280688,31.0,0.274720,1,2.0,2,114,9807,1,0,2169983,0,5.000000,0,4.499810,5.993961,0,0,0,4.0,68.0,47.0


val: первые 10 строк


,qid,item_id,bm25_all,bm25_title,bm25_params,bm25_desc,bm25_rel,rank_pool,rank_global,in_pool,cov_title,cov_all,mc_share_top20,dist_km,same_loc,filter_text_ok,rating_ok,filter_ok,pool_size,target,cos_full,cos_title,rank_dense,cos_gap,n_pos_total,q_n_terms,q_len_words,search_category,loc_n_items,geo_known,has_filter,item_microcat_id,is_remote_text,item_rating,rating_isna,reviews_log,price_log,price_is_symbolic,item_is_phone_hidden,item_is_message_forbidden,title_len,params_len,desc_len
0,tr_da53db2835a7cdcf,2d131992051cb989,0.000000,0.000000,0.000000,0.000000,0.000000,200.0,50.0,1,0.000000,0.000000,1.0,15.201652,0,1,1,1,14693,0,0.423893,0.328051,71.0,0.271773,1,3.0,4,114,28507,1,1,44728,0,5.000000,0,3.610918,0.000000,1,0,0,5.0,197.0,236.0
1,tr_da53db2835a7cdcf,33a7e4771a5d8853,8.021322,0.000000,0.000000,7.530546,0.370010,90.0,50.0,1,0.000000,0.666667,0.0,15.970729,1,0,1,0,14693,0,0.228741,0.317202,100.0,0.466925,1,3.0,4,114,28507,1,1,2303428,0,4.680412,0,4.584968,0.000000,1,0,0,5.0,22.0,115.0
2,tr_da53db2835a7cdcf,048dabd0ee217c21,14.453650,0.000000,12.842342,9.124809,0.666723,56.0,50.0,1,0.000000,1.000000,1.0,14.276658,0,1,1,1,14693,0,0.381132,0.304776,100.0,0.314534,1,3.0,4,114,28507,1,1,44728,0,5.000000,0,2.995732,6.396930,0,0,0,4.0,620.0,182.0
3,tr_da53db2835a7cdcf,7c34931b9fd2dbbf,16.558109,2.553852,6.071312,15.806046,0.763799,33.0,50.0,1,0.333333,1.000000,1.0,15.368428,1,1,1,1,14693,0,0.477032,0.360562,46.0,0.218634,1,3.0,4,114,28507,1,1,44728,0,5.000000,0,3.295837,10.817797,0,0,0,7.0,173.0,289.0
4,tr_da53db2835a7cdcf,ef11d1ec8cbced91,6.385192,0.000000,4.294514,2.579087,0.294538,129.0,50.0,1,0.000000,0.666667,0.0,13.003131,1,0,1,0,14693,0,-0.057866,0.056939,100.0,0.753532,1,3.0,4,114,28507,1,1,2058735,0,4.940476,0,4.442651,5.525453,0,0,0,3.0,213.0,149.0
5,tr_da53db2835a7cdcf,5280e8d77a295be9,9.017302,0.000000,0.000000,8.798950,0.415953,78.0,50.0,1,0.000000,0.666667,0.0,11.090480,0,0,1,0,14693,0,0.003449,0.085837,100.0,0.692217,1,3.0,4,114,28507,1,1,1288223,0,4.815217,0,4.532599,10.819798,0,0,0,4.0,101.0,246.0
6,tr_da53db2835a7cdcf,84fd6c4c3860d4bf,20.620707,14.454041,18.817001,20.282303,0.951200,200.0,33.0,0,1.000000,1.000000,1.0,624.337097,0,1,1,1,14693,0,0.605511,0.585175,100.0,0.090154,1,3.0,4,114,28507,1,1,44728,0,4.872093,0,5.153292,6.908755,0,0,0,5.0,312.0,112.0
7,tr_da53db2835a7cdcf,127d4add6639840a,15.341412,0.000000,12.842342,12.144196,0.707674,45.0,50.0,1,0.000000,1.000000,1.0,23.994780,0,1,1,1,14693,0,0.377730,0.343672,100.0,0.317936,1,3.0,4,114,28507,1,1,44728,0,5.000000,0,2.995732,6.216606,0,0,0,4.0,620.0,204.0
8,tr_da53db2835a7cdcf,3acec4bb65863027,7.314483,3.337795,2.964532,7.321335,0.337405,103.0,50.0,1,0.333333,0.666667,0.0,2.251493,1,0,1,0,14693,0,0.146117,0.267793,100.0,0.549549,1,3.0,4,114,28507,1,1,44725,0,5.000000,0,2.079442,9.210441,0,0,0,4.0,164.0,214.0
9,tr_da53db2835a7cdcf,347a3be3dab07577,20.979952,14.030869,17.093277,21.402603,0.967771,200.0,15.0,0,0.666667,1.000000,1.0,713.367554,0,1,1,1,14693,0,0.650306,0.619219,100.0,0.045360,1,3.0,4,114,28507,1,1,44728,0,4.964286,0,4.727388,6.908755,0,0,0,3.0,249.0,237.0


In [13]:
print('Статистика признаков: train vs val (среднее)')
cmp = pd.DataFrame({'train_mean': train_df[FEATURES].mean(numeric_only=True),
                    'val_mean': val_df[FEATURES].mean(numeric_only=True),
                    'train_nan_%': train_df[FEATURES].isna().mean() * 100,
                    'val_nan_%': val_df[FEATURES].isna().mean() * 100})
display(cmp.round(3))

Статистика признаков: train vs val (среднее)


,train_mean,val_mean,train_nan_%,val_nan_%
bm25_all,5.663,6.794,0.000,0.000
bm25_title,2.409,2.342,0.000,0.000
bm25_params,2.450,2.599,0.000,0.000
bm25_desc,5.099,6.012,0.000,0.000
bm25_rel,0.401,0.420,0.000,0.000
rank_pool,130.168,130.144,0.000,0.000
rank_global,45.521,45.537,0.000,0.000
in_pool,0.841,0.851,0.000,0.000
cov_title,0.228,0.174,0.000,0.000
cov_all,0.462,0.462,0.000,0.000


### Средние признаков у релевантных и нерелевантных кандидатов (val)

Грубая проверка, какие признаки разделяют классы. Если у `target=1` признак заметно выше или ниже, CatBoost сможет его использовать.

In [14]:
display(val_df.groupby('target')[FEATURES].mean(numeric_only=True).T.round(3))

target,0,1
bm25_all,6.764,15.250
bm25_title,2.324,7.476
bm25_params,2.590,5.326
bm25_desc,5.987,13.300
bm25_rel,0.418,0.836
rank_pool,130.500,27.136
rank_global,45.583,32.259
in_pool,0.851,0.963
cov_title,0.173,0.476
cov_all,0.461,0.858


### Пример одного val-запроса: кандидаты в порядке BM25 (с косинусами)

In [15]:
ex_q = val_df.loc[val_df.target == 1, 'qid'].iloc[0]
qrow = queries.set_index('qid').loc[ex_q]
print(f'запрос: «{qrow.search_query}» | фильтр: «{qrow.search_infm_params_text}» | локация: {qrow.search_location_id}')
ex = (val_df[val_df.qid == ex_q].sort_values('bm25_all', ascending=False)
      .merge(items[['item_id', 'title_norm']], on='item_id'))
display(ex[['title_norm', 'target', 'bm25_all', 'cos_full', 'cos_title', 'rank_pool', 'rank_dense', 'dist_km',
            'same_loc', 'filter_ok', 'item_rating']].head(15))

запрос: «остекление лоджий по ключ» | фильтр: «Тип услуги Окна и балконы Вид услуги Ремонт и отделка» | локация: 653240


,title_norm,target,bm25_all,cos_full,cos_title,rank_pool,rank_dense,dist_km,same_loc,filter_ok,item_rating
0,Остекление балконов остекление лоджий под ключ,0,21.732519,0.617484,0.607606,200.0,100.0,633.713989,0,1,5.000000
1,Остекление балконов и лоджий под ключ,0,21.678635,0.645128,0.625811,0.0,2.0,23.994780,0,1,4.819445
2,Остекление балконов и лоджий под ключ,0,21.658718,0.607369,0.625811,200.0,100.0,651.663025,0,1,4.819445
3,"Балкон под ключ, лоджия под ключ, остекление балкона и л...",0,21.620907,0.601937,0.557533,200.0,100.0,1355.797241,0,1,5.000000
4,"Окна, Балконы, Остекление, Отделка балконов, Двери",0,21.381588,0.497713,0.496719,200.0,100.0,1050.145020,0,1,4.915254
5,Отделка и остекление балконов под ключ СПБ и ло,0,21.262606,0.589458,0.554595,1.0,15.0,11.090480,0,1,5.000000
6,Остекление балконов и лоджий под ключ,0,21.246700,0.634066,0.625811,2.0,6.0,24.320650,0,1,5.000000
7,Остекление утепления ремонт балконов и лоджии,0,21.230391,0.612782,0.549593,200.0,100.0,630.262939,0,1,4.909091
8,Теплое остекление балконов без сквозняков,0,21.227669,0.599272,0.514351,200.0,100.0,644.856323,0,1,5.000000
9,Отделка балконов и лоджий под ключ,0,21.218248,0.534121,0.494834,200.0,100.0,651.583252,0,1,5.000000
